# LLM feature extraction from Item 1.05 disclosures

Runs **free** on Google Colab. Before running: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

A small open model (Qwen 2.5 3B Instruct) reads the Item 1.05 section of every incident and returns structured
features as JSON. The output `llm_features.csv` downloads at the end; add it to `data/processed/` in the repo
and rerun `python -m src.analysis`.

Takes about 5–10 minutes. Nothing is installed or stored on your own computer.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import json, re
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
EVENTS_URL = "https://raw.githubusercontent.com/vaidehidoye/cyber-disclosure-event-study/main/data/processed/events.csv"

events = pd.read_csv(EVENTS_URL)
print(len(events), "incidents;", "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (switch runtime to T4 GPU!)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")

## Prompt and output schema
Each field is defined precisely so the answers can be compared with the keyword features and hand-coding.

In [ ]:
SYSTEM = "You are a careful financial analyst who extracts facts from SEC filings. Answer only with valid JSON."

PROMPT = """Read this Item 1.05 (Material Cybersecurity Incident) disclosure from a Form 8-K and return a JSON object with exactly these keys:

- "breach_type": one of "ransomware", "data_theft", "unauthorized_access", "payment_fraud", "other".
  Use "ransomware" if systems were encrypted or a ransom/extortion is mentioned, even without the word ransomware.
- "personal_data_involved": true if customer, employee or personal data was or may have been accessed or taken, else false.
- "operations_disrupted": true if business operations, systems or services were disrupted, taken offline or impaired, else false.
- "materiality": the company's own assessment of the financial impact: "material" (it says the impact is or is reasonably
  likely to be material), "not_material" (it says the impact is not, or is not reasonably likely to be, material),
  or "undetermined" (it says it cannot yet determine the impact).
- "threat_actor_named": true if a specific threat actor, group or nation-state is named or attributed, else false.
- "severity": integer 1-5, how severe the incident sounds for the business (1 = minor, 5 = very severe).
- "vagueness": integer 1-5, how vague the disclosure is (1 = concrete facts: dates, systems, data, impact; 5 = generic boilerplate).

Disclosure:
\"\"\"{text}\"\"\"

JSON:"""

FIELDS = ["breach_type", "personal_data_involved", "operations_disrupted", "materiality", "threat_actor_named", "severity", "vagueness"]


def ask(text: str) -> str:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": PROMPT.format(text=text[:6000])}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(inputs, max_new_tokens=200, do_sample=False)  # greedy = reproducible
    return tokenizer.decode(output[0, inputs.shape[1]:], skip_special_tokens=True)


def parse(answer: str) -> dict | None:
    match = re.search(r"\{.*\}", answer, flags=re.DOTALL)
    try:
        data = json.loads(match.group(0)) if match else None
    except json.JSONDecodeError:
        return None
    return data if data and all(f in data for f in FIELDS) else None


print(ask(events["text"].iloc[0]))

## Run on all incidents

In [ ]:
rows = []
for i, ev in events.iterrows():
    result = parse(ask(ev["text"])) or parse(ask(ev["text"] + "\n\nRemember: reply with the JSON object only."))
    rows.append({"accession": ev["accession"], **(result or {f: None for f in FIELDS})})
    print(f"{i + 1}/{len(events)} {ev['company'][:40]:40} {result}")

llm = pd.DataFrame(rows)
print("Failed to parse:", llm["breach_type"].isna().sum())
llm.head()

## Quick look and download

In [ ]:
for col in ["breach_type", "materiality", "personal_data_involved", "operations_disrupted", "threat_actor_named"]:
    print(llm[col].value_counts(dropna=False), "\n")
print(llm[["severity", "vagueness"]].describe())

llm.to_csv("llm_features.csv", index=False)
from google.colab import files
files.download("llm_features.csv")